# P14′ (3B): CONSTRUCT trên benchmark chuẩn (OV-MERD+, MER2025-FG) và EmoS-MELD mới, Kaggle **GPU T4 ×2**
Thiết kế và tiêu chí đặt trước: `explore/ovmer/p14_score.py`. Model: `Qwen/Qwen2.5-Omni-3B`; 48 mẫu mỗi clip. Hai tiến trình song song, mỗi GPU một nửa số clip.
Cần secret **HF_TOKEN** (MER2024 không cần; MER2026 cần); nếu zip MER2026 có mật khẩu thì thêm secret **MER2026_ZIP_PW**. Accelerator **GPU T4 x2**; Internet **On**.

In [ ]:
HOURS = 8.5
MODEL, QUANT4, TAG, NSAMP = 'Qwen/Qwen2.5-Omni-3B', False, '3B', 48
KOUT, WORK = '/kaggle/working', '/tmp/p14'
MELD_TAR = 'https://huggingface.co/datasets/declare-lab/MELD/resolve/main/MELD.Raw.tar.gz'

In [ ]:
import os, subprocess
sh = lambda cmd, check=True: subprocess.run(cmd, shell=True, check=check)
os.makedirs(WORK, exist_ok=True)
sh(f'rm -rf {WORK}/icme && git clone -q --depth 1 -b main https://github.com/ptrnghieu/icme.git {WORK}/icme')
sh(f'rm -rf {WORK}/affectgpt && git clone -q --depth 1 https://github.com/zeroQiaoba/AffectGPT {WORK}/affectgpt')
sh(f'rm -rf {WORK}/emos && git clone -q --depth 1 https://github.com/NLP2CT/EmoS {WORK}/emos')
print('code commit:', subprocess.run(f'git -C {WORK}/icme log -1 --format=%h_%cd', shell=True, capture_output=True, text=True).stdout)
sh('pip -q install -U huggingface_hub qwen-omni-utils openpyxl' + (' bitsandbytes' if QUANT4 else ''), check=False)
from kaggle_secrets import UserSecretsClient
try:
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
except Exception:
    raise SystemExit('HF_TOKEN secret is not attached to THIS notebook: Add-ons -> Secrets -> tick HF_TOKEN, then Save & Run All again.') from None
try:
    os.environ['MER2026_ZIP_PW'] = UserSecretsClient().get_secret('MER2026_ZIP_PW')   # only needed if the zip is encrypted; never printed
except Exception:
    pass
os.environ['HF_HOME'] = f'{WORK}/hf'
import torch, transformers; print('GPUs:', torch.cuda.device_count(), '| transformers', transformers.__version__)

In [ ]:
from huggingface_hub import hf_hub_download, snapshot_download
Z26 = hf_hub_download(repo_id='MERChallenge/MER2026', repo_type='dataset', filename='video_7z/video_track2_train_human/video_split.zip',
                      local_dir=f'{WORK}/MER2026', token=os.environ['HF_TOKEN'])
print('MER2026 human video zip', f'{os.path.getsize(Z26) / 1e9:.2f} GB')
snapshot_download(MODEL)  # download once, before the two worker processes start
ITEMS = f'{KOUT}/p14_items.jsonl'
subprocess.run(['python', f'{WORK}/icme/explore/ovmer/p13_prepare.py', '--p14', '--affectgpt', f'{WORK}/affectgpt', '--emos', f'{WORK}/emos',
                '--mer26zip', Z26, '--meld_tar', MELD_TAR, '--work', f'{WORK}/media', '--out', ITEMS], check=False)

In [ ]:
R = f'{WORK}/icme/explore/ovmer/p13_mbr.py'; procs = []
for g in range(2):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(g))
    cmd = ['python', R, 'run', '--items', ITEMS, '--out', f'{KOUT}/p14_{TAG}_out_{g}.jsonl', '--hours', str(HOURS), '--model', MODEL,
           '--n_samples', str(NSAMP), '--shard', f'{g}/2'] + (['--quant4'] if QUANT4 else [])
    procs.append(subprocess.Popen(cmd, env=env, stdout=open(f'{KOUT}/p14_{TAG}_log_{g}.txt', 'w'), stderr=subprocess.STDOUT))
import time
while any(p.poll() is None for p in procs):
    time.sleep(300)
    for g in range(2):
        lines = [l for l in open(f'{KOUT}/p14_{TAG}_log_{g}.txt', errors='ignore') if 'clips,' in l or 'fail' in l or 'Error' in l]
        print(f'GPU{g}:', lines[-1].strip() if lines else '(starting)', flush=True)
print('return codes:', [p.returncode for p in procs])

In [ ]:
OUT = f'{KOUT}/p14_{TAG}_out.jsonl'
with open(OUT, 'w') as f:
    for g in range(2):
        p = f'{KOUT}/p14_{TAG}_out_{g}.jsonl'
        if os.path.exists(p):
            f.write(open(p).read())
subprocess.run(['python', f'{WORK}/icme/explore/ovmer/p14_score.py', '--items', ITEMS, '--outs', OUT, '--names', TAG, '--seeds', str(NSAMP // 16),
                '--mapping', f'{WORK}/affectgpt/AffectGPT-RL/emotion_wheel/wheel_mapping.npz', '--report', f'{KOUT}/p14_{TAG}_report.md'], check=False)
print(open(f'{KOUT}/p14_{TAG}_report.md').read())